# Chessboard Dataset Creation

Downloads the Lichess puzzle database, balances/splits it, renders the Task 1 / Task 2 / Task 3 image datasets **in parallel** across all available CPU cores (via `generate_datasets_parallel` in `src/data/generation.py`), and uploads each task's dataset to the Hugging Face Hub.

In [ ]:
!pip install -q chess cairosvg huggingface_hub datasets scikit-learn pandas tqdm zstandard python-dotenv

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, login
from datasets import load_dataset

In [3]:
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}


In [4]:
if CONFIG["colab"]:
    repo_dir = Path(CONFIG["repo_dir"])
    if repo_dir.exists():
        subprocess.run(["rm", "-rf", str(repo_dir)], check=True)

    auth_url = "https://"
    repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_dir)],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f"Git clone failed: {result.stderr}"

    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))

REPO_ROOT = Path(".").resolve()
print("Setup Complete. REPO_ROOT:", REPO_ROOT)

Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject


In [ ]:
sys.path.insert(0, str(repo_dir))
sys.path.insert(0, str(repo_dir / "src" / "data"))

from src.data.generation import generate_datasets_parallel
from src.data.utilities import load_lichess_csv

## Authenticate with Hugging Face

- **Colab:** add a token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar; needs **write** access to the `bdatm-project` org) and toggle "Notebook access" on for it.
- **Local:** create a `.env` file (already `.gitignore`d) in the repo root with a line `HF_TOKEN=hf_...`, or export `HF_TOKEN` as a regular environment variable before launching Jupyter. If neither is set, this cell falls back to an interactive login prompt.

Both paths are driven by the same `CONFIG["colab"]` flag used for the repo setup above.

In [ ]:
hf_token = None

if CONFIG["colab"]:
    # Colab secrets manager (key icon in the left sidebar)
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    # Local run: load a repo-root .env file (if present) into os.environ,
    # then read HF_TOKEN the same way as any other environment variable.
    from dotenv import load_dotenv
    load_dotenv(repo_dir / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.")
    login()

## Load the Lichess puzzle database

In [ ]:
csv_path = Path("lichess_db_puzzle.csv")
zst_path = Path("lichess_db_puzzle.csv.zst")
lichess_zst_url = "https://database.lichess.org/lichess_db_puzzle.csv.zst"

if not csv_path.exists():
    # 1. Download the compressed archive if not already present
    if not zst_path.exists():
        print("Downloading Lichess puzzles database (~250 MB compressed)...")
        if CONFIG["colab"]:
            !wget -q --show-progress {lichess_zst_url}
        else:
            import urllib.request
            urllib.request.urlretrieve(lichess_zst_url, zst_path)

    # 2. Decompress to CSV and remove the archive
    print("Decompressing archive (~1.5 GB CSV)...")
    if CONFIG["colab"]:
        # Colab: use the system zstd tool
        if shutil.which("zstd") is None and shutil.which("unzstd") is None:
            print("Installing zstd decompression tool...")
            !apt-get update -qq && apt-get install -y zstd -qq
        !unzstd -f --rm lichess_db_puzzle.csv.zst -o lichess_db_puzzle.csv
        !rm -f lichess_db_puzzle.csv.zst.*
    else:
        # Local: decompress in pure Python (no system zstd binary required)
        import zstandard as zstd
        with open(zst_path, "rb") as compressed, open(csv_path, "wb") as decompressed:
            zstd.ZstdDecompressor().copy_stream(compressed, decompressed)
        zst_path.unlink()

    print("Decompression complete!")
else:
    print(f"'{csv_path}' already exists.")

In [ ]:
# Number of puzzles to sample from the CSV, and how many White-to-move /
# Black-to-move puzzles to keep (balanced 50/50). The values below are a
# quick smoke-test size; raise them for a real run (MAX_SAMPLES must stay
# >= N_PER_TURN * 2).
MAX_SAMPLES = 100
N_PER_TURN = 20

df_raw = load_lichess_csv(csv_path=csv_path, max_samples=MAX_SAMPLES)

# Extract the active turn from FEN for balancing ('w' or 'b')
df_raw["turn"] = df_raw["FEN"].apply(
    lambda f: f.split(" ")[1] if len(f.split(" ")) > 1 else "w"
)

df_white = df_raw[df_raw["turn"] == "w"].sample(n=N_PER_TURN, random_state=42)
df_black = df_raw[df_raw["turn"] == "b"].sample(n=N_PER_TURN, random_state=42)

df_balanced = (
    pd.concat([df_white, df_black])
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)

print(f"Balanced dataset loaded: {len(df_balanced)} total puzzles")
print(df_balanced["turn"].value_counts())

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(
    df_balanced, test_size=0.20, random_state=42, stratify=df_balanced["turn"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["turn"]
)

splits = {"train": train_df, "validation": val_df, "test": test_df}
print(f"Split sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

## Generate Task 1 / Task 2 / Task 3 datasets (in parallel)

The same `splits` dict is reused for all three tasks (a given puzzle is independent across tasks). `generate_datasets_parallel` flattens every (task, split, row) combination into one work list and renders it with a single pool of worker processes — instead of looping task-by-task, split-by-split, sample-by-sample as before — so all 9 (task × split) combinations render concurrently rather than one after another.

In [ ]:
tasks = ["task1", "task2", "task3"]
task_splits = {task: splits for task in tasks}

generate_datasets_parallel(
    task_splits=task_splits,
    output_root=".",
    image_size=512,
)

## Sanity-check the generated datasets

Quick text-only check: sample counts per (task, split), plus confirmation every image file referenced by `metadata.jsonl` actually exists on disk. (For a visual look at the rendered boards, see `main.ipynb`'s "Dataset Displaying" section.)

In [ ]:
splits_to_check = ["train", "validation", "test"]

print("=== DATASET STATISTICS & INTEGRITY CHECK ===")
for task in tasks:
    for split_name in splits_to_check:
        split_dir = Path(f"dataset_{task}/{split_name}")
        jsonl_path = split_dir / "metadata.jsonl"

        if not jsonl_path.exists():
            print(f"[{task}/{split_name}] metadata.jsonl not found.")
            continue

        with open(jsonl_path, "r", encoding="utf-8") as f:
            records = [json.loads(line) for line in f]

        missing = [
            r["sample_id"] for r in records
            if not (split_dir / r["file_name"]).exists()
            or (r["file_name_t1"] and not (split_dir / r["file_name_t1"]).exists())
        ]

        status = "OK" if not missing else f"MISSING FILES for: {missing}"
        print(f"[{task}/{split_name}] {len(records)} sample(s) — {status}")

print("\nIntegrity check complete.")

## Upload each generated dataset to Hugging Face

In [ ]:
ORGANIZATION_NAME = "bdatm-project"
api = HfApi()

print("=== UPLOADING TO HUGGING FACE ===")
for task in tasks:
    local_directory = f"dataset_{task}"
    repo_id = f"{ORGANIZATION_NAME}/{local_directory}"

    print(f"Uploading '{local_directory}' to '{repo_id}'...")
    api.upload_folder(
        folder_path=local_directory,
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=f"Upload generated {task} dataset",
    )

print("\nAll datasets have been synchronized successfully to Hugging Face!")

## Verify: load one dataset back from the Hub

In [ ]:
repo_id_to_check = f"{ORGANIZATION_NAME}/dataset_task1"
print(f"Loading dataset from Hugging Face: {repo_id_to_check}")

dataset = load_dataset(repo_id_to_check)
print("\nDataset loaded successfully!")
print(dataset)

sample = dataset["train"][0]
print("\n--- First training sample ---")
for key, value in sample.items():
    if key == "image":
        print(f" - {key}: PIL image, size {value.size}")
    else:
        print(f" - {key}: {value}")